# BAN 612 — Week 6 In-Class Practice Notebook
## Exploratory Analysis with SaaS Customer Data

This notebook continues with the Week 5 SaaS dataset, but the objective is different. Week 5 focused on reshaping, joining, and preparing data. Week 6 focuses on **exploratory analysis**: using prepared data to ask better business questions, identify patterns, compare segments, and communicate defensible findings.

**Business context:** A SaaS company wants to understand customer value, engagement, support burden, payment behavior, and churn risk signals.

## 0. Setup

The code below imports the libraries used in this notebook.

- `pandas` is used for tabular data analysis.
- `numpy` is used for numerical operations.
- `matplotlib.pyplot` is used for basic charts.
- `Path` helps make file paths work across different computers.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Make pandas display a reasonable number of columns in outputs.
pd.set_option("display.max_columns", 80)

# Use a default figure size for charts.
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Load the SaaS data

The Week 5 dataset has five main files:

- `customers.csv`: one row per customer
- `subscriptions.csv`: customer plan and recurring revenue information
- `usage.csv`: one row per customer per week
- `invoices.csv`: billing and payment records
- `tickets.csv`: support interactions



In [7]:
#Same as Week 5# Set the data director and load the data (all 6 files)
data_dir = Path('BAN612_Week5_SaaS_Data')

customers = pd.read_csv(data_dir / "customers.csv")
subscriptions = pd.read_csv(data_dir / "subscriptions.csv")
invoices = pd.read_csv(data_dir / "invoices.csv")
usage = pd.read_csv(data_dir / "weekly_usage.csv")
tickets = pd.read_csv(data_dir / "support_tickets.csv")
usage_wide = pd.read_csv(data_dir / "usage_wide_demo.csv")

# Quick row counts. This is not the main analysis, but it gives us orientation.
for name, df in {
    'customers': customers,
    'subscriptions': subscriptions,
    'invoices': invoices,
    'usage': usage,
    'tickets': tickets,
    'usage_wide': usage_wide
}.items():
    print(f'{name:15s}', df.shape)

customers       (140, 7)
subscriptions   (158, 8)
invoices        (1142, 8)
usage           (5928, 7)
tickets         (300, 7)
usage_wide      (134, 7)


## 2. Minimal preparation before EDA

Even though Week 6 is about exploration, we still need basic preparation:

1. Convert date columns to datetime.
2. Check row meaning and key uniqueness.
3. Avoid interpreting text dates as actual dates.

`errors="coerce"` tells pandas to convert unparseable values to `NaT` rather than crashing. That makes bad dates visible during validation.

In [8]:
# Convert date columns to datetime.
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")

subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"], errors="coerce")
subscriptions["end_date"] = pd.to_datetime(subscriptions["end_date"], errors="coerce")

usage["week_start"] = pd.to_datetime(usage["week_start"], errors="coerce")

invoices["invoice_date"] = pd.to_datetime(invoices["invoice_date"], errors="coerce")
invoices["paid_date"] = pd.to_datetime(invoices["paid_date"], errors="coerce")

tickets["opened_at"] = pd.to_datetime(tickets["opened_at"], errors="coerce")
tickets["closed_at"] = pd.to_datetime(tickets["closed_at"], errors="coerce")

## 3. Time coverage: what period does each source cover?

Before exploring trends, first identify the time period of each table. Different tables may cover different periods, and that affects interpretation.

In [9]:
time_coverage = pd.DataFrame({
    "source": ["customers", "subscriptions", "weekly_usage", "invoices", "support_tickets"],
    "date_column": ["signup_date", "start_date", "week_start", "invoice_date", "opened_at"],
    "min_date": [
        customers["signup_date"].min(),
        subscriptions["start_date"].min(),
        usage["week_start"].min(),
        invoices["invoice_date"].min(),
        tickets["opened_at"].min()
    ],
    "max_date": [
        customers["signup_date"].max(),
        subscriptions["start_date"].max(),
        usage["week_start"].max(),
        invoices["invoice_date"].max(),
        tickets["opened_at"].max()
    ]
})

time_coverage

,source,date_column,min_date,max_date
0,customers,signup_date,2024-01-04 00:00:00,2025-06-28 00:00:00
1,subscriptions,start_date,2024-01-07 00:00:00,2025-11-25 00:00:00
2,weekly_usage,week_start,2025-01-06 00:00:00,2025-12-29 00:00:00
3,invoices,invoice_date,2025-01-01 00:00:00,2025-12-04 00:00:00
4,support_tickets,opened_at,2025-01-01 08:00:00,2025-12-31 12:00:00


## 4. Build a customer-level analysis table

EDA often requires changing the unit of analysis. The raw tables are at different units:

- customer table: customer-level
- subscriptions: subscription-level
- usage: customer-week-level
- invoices: customer-invoice-level
- support tickets: customer-ticket-level

For customer segmentation, we need a **customer-level table**. We will aggregate usage, billing, and support data to one row per customer.

In [10]:
# Latest subscription per customer. The tail(1) keeps only the last entry. Since we sort by start_date, this would give the last row
latest_subscription = (
    subscriptions
    .sort_values(["customer_id", "start_date"])
    .groupby("customer_id", as_index=False)
    .tail(1)
    [["customer_id", "plan_name", "billing_cycle", "mrr", "subscription_status", "start_date"]]
)

latest_subscription.head()

,customer_id,plan_name,billing_cycle,mrr,subscription_status,start_date
0,C1000,Growth,Annual,245.00,Active,2024-07-30
1,C1001,Growth,Monthly,207.23,Active,2024-12-10
2,C1002,Starter,Monthly,89.78,Churned,2024-06-01
3,C1003,Starter,Annual,100.13,Churned,2024-02-12
5,C1004,Growth,Monthly,256.86,Active,2025-01-15


In [11]:
# Usage metrics per customer.
usage["logins_per_active_user"] = (
    usage["logins"] / usage["active_users"].replace(0, np.nan)
)

usage_by_customer = (
    usage
    .groupby("customer_id", as_index=False)
    .agg(
        weeks_observed=("week_start", "nunique"),
        max_active_users=,
        avg_logins=,
        avg_reports=,
        avg_api_calls=,
        max_storage_gb=,
        avg_logins_per_active_user=("logins_per_active_user", "mean")
    )
)

usage_by_customer.head()

SyntaxError: invalid syntax (1321786743.py, line 11)

In [12]:
# Invoice metrics per customer.
invoices["days_to_pay"] = (invoices["paid_date"] - invoices["invoice_date"]).dt.days
invoices["is_unpaid"] = ((invoices["amount_due"] > 0) & (invoices["amount_paid"] == 0))

invoice_by_customer = (
    invoices
    .groupby(, as_index=False)
    .agg(
        invoice_count=
        total_due=,
        total_paid=,
        avg_days_to_pay=,
        unpaid_invoices=,
        total_refunds=
    )
)

invoice_by_customer["payment_realization"] = (
    invoice_by_customer["total_paid"] / invoice_by_customer["total_due"].replace(0, np.nan)
)

invoice_by_customer.head()

SyntaxError: invalid syntax (2945493632.py, line 7)

In [ ]:
# Support metrics per customer.
tickets["resolution_hours"] = 

support_by_customer = (
    tickets
    .groupby("customer_id", as_index=False)
    .agg(
        ticket_count=,
        high_priority_tickets=,
        avg_csat=,
        avg_resolution_hours=
    )
)

support_by_customer.head()

In [ ]:
# Merge everything into a customer-level analysis table.
customer360 = (
    customers
    .merge
)

# Customers with no support tickets will have missing support metrics.
customer360["ticket_count"] = customer360["ticket_count"].fillna(0)
customer360["high_priority_tickets"] = 

customer360.shape

## 5. Univariate EDA: what does one variable look like?

Start with one variable at a time. For SaaS, `mrr` is a key business variable.

Things to look for:

- Is the distribution symmetric or skewed?
- Are there extreme customers?
- Are mean and median very different?
- Does the chart suggest different customer types?

In [ ]:
customer360["mrr"].describe()

In [ ]:
plt.hist(customer360["mrr"].dropna(), bins=20)
plt.title("Distribution of Monthly Recurring Revenue")
plt.xlabel("MRR ($)")
plt.ylabel("Number of customers")
plt.show()

## 6. Categorical EDA: how are customers distributed across groups?

Before comparing customer groups, inspect the category values. This helps catch inconsistent labels and small groups.

In [ ]:
#Find for customer_size

In [ ]:
#Find for plan_name

## 7. Compare groups: MRR by company size and plan

Group comparison helps move beyond averages. Here we compare median MRR rather than average MRR because MRR is usually skewed.

In [ ]:
size_summary = (
    customer360
    .groupby
    .agg(
        customers=
        median_mrr=
        avg_mrr=
        median_active_users=
    )
    .sort_values("median_mrr", ascending=False)
    .reset_index(drop=True)
)

size_summary

In [ ]:
size_summary.plot(
    x="company_size",
    y="median_mrr",
    kind="bar",
    legend=False
)
plt.title("Median MRR by Customer Size")
plt.xlabel("Customer size")
plt.ylabel("Median MRR ($)")
plt.xticks(rotation=35, ha="right")
plt.show()

## 8. Time-series EDA: usage trend over time

For the usage table, we can aggregate weekly rows into months. The goal is not forecasting yet; the goal is to see whether engagement is stable, rising, falling, or seasonal.

In [ ]:
usage["month"] = usage["week_start"].dt.to_period("M").dt.to_timestamp() #Converting to time stamp helps in plotting later.

monthly_usage = (
    usage
    .groupby("month", as_index=False)
    .agg(
        total_logins=
        active_customers=
        avg_active_users=
        total_reports=
    )
)

monthly_usage.head()

In [ ]:
plt.plot(monthly_usage["month"], monthly_usage["total_logins"], marker="o")
plt.title("Total Logins by Month")
plt.xlabel("Month")
plt.ylabel("Total logins")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
#Try plotting both logins and users in the same plot

## 9. Relationship EDA: usage versus revenue

A common business question is whether higher-value customers also use the product more.

A scatterplot is useful because it shows individual customers, not just grouped averages.

In [ ]:
plt.scatter(customer360["avg_active_users"], customer360["mrr"], alpha=0.65)
plt.title("Average Active Users vs MRR")
plt.xlabel("Average active users per week")
plt.ylabel("MRR ($)")
plt.show()

In [ ]:
customer360[["mrr", "avg_active_users", "avg_logins", "avg_reports", "avg_api_calls"]].corr()

## 10. Support burden and satisfaction

Support tickets are not automatically bad. A high-value customer may create more tickets because they use the product heavily. That is why we should compare support burden relative to usage or revenue, not only raw ticket counts.

In [ ]:
customer360["tickets_per_1000_logins"] = (
    1000 * customer360["ticket_count"] /
    (customer360["avg_logins"] * customer360["weeks_observed"]).replace(0, np.nan)
)

#Group by Plan
support_summary = (
    customer360
    .groupby
    .agg(
        customers=
        avg_tickets=
        avg_csat=
        median_tickets_per_1000_logins=
    )
)

support_summary

In [ ]:
plt.scatter(customer360["ticket_count"], customer360["avg_csat"], alpha=0.70)
plt.title("Support Tickets vs Average CSAT")
plt.xlabel("Support ticket count")
plt.ylabel("Average CSAT")
plt.show()

## 11. Identify outliers and explain why they matter

An outlier is not automatically an error. In business analytics, an outlier may be:

- a valuable enterprise customer,
- a struggling customer needing attention,
- a data-quality issue,
- or a business process exception.

Let's identify customers with unusually high MRR and unusually high support burden.

In [ ]:
high_mrr_cutoff = customer360["mrr"].quantile(0.90)
high_support_cutoff = customer360["tickets_per_1000_logins"].quantile(0.90)

outlier_customers = customer360[
    (customer360["mrr"] >= high_mrr_cutoff) |
    (customer360["tickets_per_1000_logins"] >= high_support_cutoff)
].copy()

#Show some high MRR customers and their features

## 12. Segment customers using value and engagement

Segmentation helps translate EDA into management action. Here we create simple terciles for value and engagement. This is not a predictive model; it is an interpretable descriptive segmentation.

In [ ]:
#Divide into 3 groups based on MRR and logins
customer360["value_segment"] = pd.qcut(
    customer360["mrr"].rank(method="first"),
    q=3,
    labels=["Low value", "Mid value", "High value"]
) #method tells you how ties are broken

customer360["engagement_segment"] = pd.qcut(
    customer360["avg_logins_per_active_user"].rank(method="first"),
    q=3,
    labels=["Low engagement", "Mid engagement", "High engagement"]
)

segment_matrix = pd.crosstab(
    customer360["value_segment"],
    customer360["engagement_segment"]
)

segment_matrix

In [ ]:
plt.imshow(segment_matrix.values)
plt.xticks(range(segment_matrix.shape[1]), segment_matrix.columns, rotation=25, ha="right")
plt.yticks(range(segment_matrix.shape[0]), segment_matrix.index)
for i in range(segment_matrix.shape[0]):
    for j in range(segment_matrix.shape[1]):
        plt.text(j, i, segment_matrix.iloc[i, j], ha="center", va="center")
plt.title("Customer Count: Value Segment x Engagement Segment")
plt.colorbar(label="Customers")
plt.show()

## 13. Create an interpretable risk flag

This is **not** a formal churn model. It is a transparent exploratory index to help prioritize follow-up.

We will flag customers that show several possible risk signals:

- low engagement intensity,
- low payment realization,
- low CSAT,
- high support burden,
- cancelled subscription status.

Do not treat the score as truth. Treat it as a way to find records worth investigating.

In [ ]:
customer360["risk_score"] = 0

customer360["risk_score"] += np.where(
    customer360["avg_logins_per_active_user"] < customer360["avg_logins_per_active_user"].quantile(0.25),
    2,
    0
)

customer360["risk_score"] += np.where(
    customer360["payment_realization"] < 0.98,
    2,
    0
)

customer360["risk_score"] += np.where(
    customer360["avg_csat"] < 3.5,
    1,
    0
)

customer360["risk_score"] += np.where(
    customer360["ticket_count"] > customer360["ticket_count"].quantile(0.75),
    1,
    0
)

customer360["risk_score"] += np.where(
    customer360["subscription_status"].eq("Cancelled"),
    3,
    0
)

customer360[["customer_id", "risk_score"]].head()

In [ ]:
top_10_risky = (
    customer360
    .sort_values(["risk_score", "mrr"], ascending=[False, False])
    [[
        "customer_id", "company_name", "plan_name", "mrr", "risk_score",
        "avg_logins_per_active_user", "payment_realization", "ticket_count", "avg_csat"
    ]]
    .head(10)
)

top_10_risky

## 14. Executive summary table

EDA should end with a concise set of findings, not just charts. Here we create a segment-level table that a manager could read.

In [ ]:
segment_summary = (
    customer360
    .groupby(["value_segment", "engagement_segment"], observed=False)
    .agg(
        customers=("customer_id", "nunique"),
        median_mrr=("mrr", "median"),
        avg_risk_score=("risk_score", "mean"),
        avg_csat=("avg_csat", "mean"),
        median_payment_realization=("payment_realization", "median")
    )
    .reset_index()
    .sort_values(["value_segment", "engagement_segment"])
)

segment_summary

## 15. Practice exercise

Use the same `customer360` table to answer the following. Write code and then write 2–3 sentences interpreting your findings.

1. Which plan has the highest median MRR?
2. Which customer size group has the highest median engagement intensity?
3. Which industry has the highest average support tickets per customer?
4. Find the top 10 customers by `mrr` and compare their average CSAT with the rest of the customers.
5. Create your own risk rule using at least three variables. How does your top-10 list differ from the one above?
6. What is one pattern you would **not** present as a firm conclusion because the subgroup is too small?

In [ ]:
# Practice workspace
# Write your code below.

## 16. Closing thought

Exploratory analysis is not a hunt for charts. It is a disciplined process for turning prepared data into better questions, clearer comparisons, and defensible business insights.